# BM25 training and development on Colab

Mount the project and supplied training files from Drive, then generate separate BM25 candidate samples for the frozen training and development Source 1 splits. The notebook can use CuPy for the sparse BM25 score multiplication when a CUDA GPU and matching CuPy wheel are available; otherwise it uses BM25S/Numba on CPU. Text preparation and top-k sorting remain on CPU.

This is a retrieval experiment, not a replacement for the submission pipeline. It reads only supplied training files and joins ground truth after retrieval to measure the sampled link recall. It does not read test files or use external business data. Outputs go under `artifacts/bm25_pipeline/{train,development}`.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

# The Drive project should contain this repository and the supplied train TSVs.
PROJECT_DIR = Path('/content/drive/MyDrive/amazon-ml')
PACKAGE = PROJECT_DIR / 'code/business_entity_resolution'
TRAIN_DIR = PROJECT_DIR / 'student_resource/dataset/train'
ARTIFACTS_V2 = PACKAGE / 'artifacts/v2'

required = [TRAIN_DIR / name for name in (
    'train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv', 'train_ground_truth.tsv'
)]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing supplied training files:\n' + '\n'.join(missing))
if not (PACKAGE / 'bm25_pipeline/train.py').is_file():
    raise FileNotFoundError(f'BM25 project code not found under {PROJECT_DIR}')
print(f'Project: {PROJECT_DIR}')
print(f'Train data: {TRAIN_DIR}')

In [ ]:
# Use uv for reproducible dependency installation in the Colab environment.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-r', str(PACKAGE / 'requirements.txt'),
                '-r', str(PACKAGE / 'requirements-retrieval.txt')], check=True)

# GPU is optional. Install the CUDA 12 wheel when the runtime exposes a GPU.
gpu_available = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).returncode == 0
if gpu_available:
    try:
        subprocess.run(['uv', 'pip', 'install', '--system', '-r',
                        str(PACKAGE / 'requirements-bm25-gpu.txt')], check=True)
        import cupy as cp
        cp.cuda.runtime.getDeviceCount()
        BACKEND = 'cupy'
        print('CuPy CUDA sparse scoring is available.')
    except Exception as exc:
        BACKEND = 'bm25s'
        print(f'CuPy could not access the GPU ({exc}); using BM25S/Numba CPU.')
else:
    BACKEND = 'bm25s'
    print('No CUDA GPU detected; using BM25S/Numba CPU.')
print('Selected backend:', BACKEND)

## Run settings

The defaults reproduce the pilot's bounded sample size and keep output volume manageable. Increase the per-country limits for a broader experiment. The saved split is the v2 seed-42 split; if no v2 DuckDB exists, the code imports the four supplied training files and creates the same split.

In [ ]:
DOCS_PER_COUNTRY = 150_000
TARGETS_PER_COUNTRY = 20_000
CANDIDATE_CAP = 32
BATCH_SIZE = 512

def run_partition(script_name):
    command = [
        sys.executable, str(PACKAGE / 'bm25_pipeline' / script_name),
        '--train-dir', str(TRAIN_DIR),
        '--artifacts-dir', str(ARTIFACTS_V2),
        '--backend', BACKEND,
        '--docs-per-country', str(DOCS_PER_COUNTRY),
        '--targets-per-country', str(TARGETS_PER_COUNTRY),
        '--cap', str(CANDIDATE_CAP),
        '--batch-size', str(BATCH_SIZE),
    ]
    subprocess.run(command, cwd=PROJECT_DIR, check=True)

run_partition('train.py')

In [ ]:
run_partition('develop.py')

## Results

Each run writes a compressed candidate TSV and JSON report to its own `train/` or `development/` directory. The reports include the selected backend, per-country elapsed time, candidate volume, source-specific positive-link recall, and the fraction of sampled linked Source 1 entities for which every sampled true link was retained. Development metrics are descriptive for the fixed sample; they are not the official held-out model F₀.₅.